In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os
import pickle
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Configuração de diretórios
DATA_DIR = "../data"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

# 2. Confirmar que a GPU T4 está ativa
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"A usar dispositivo: {device}")


df_val = pd.read_csv(os.path.join(DATA_DIR, "dataset_validacao.csv"))
df_val = df_val.dropna(subset=['Text', 'Label']) # Limpeza por segurança

unique_labels = sorted(df_val['Label'].unique().tolist())
label_mapping = {k: v for v, k in enumerate(unique_labels)}
reverse_mapping = {v: k for k, v in label_mapping.items()}

print("Mapeamento recriado com sucesso:", label_mapping)


In [ ]:
# 1. Definição da Arquitetura (Igual ao treino)
class EndToEndDetector(nn.Module):
    def __init__(self, model_name, num_classes, hidden_size=128):
        super(EndToEndDetector, self).__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        self.fc1 = nn.Linear(self.transformer.config.hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        out = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        mask_expanded = attention_mask.unsqueeze(-1).expand(out.last_hidden_state.size()).float()
        sum_embeddings = torch.sum(out.last_hidden_state * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask
        x = self.dropout(self.relu(self.fc1(mean_pooled)))
        return self.fc2(x)

# 2. Inicializar e carregar pesos
MODEL_NAME = 'microsoft/deberta-v3-small'
model = EndToEndDetector(MODEL_NAME, num_classes=len(label_mapping)).to(device)

# Carregar o ficheiro de pesos (subm3_deberta_final.pt)
model.load_state_dict(torch.load('subm3_deberta_final_v2.pt', map_location=device))
model.eval()
print("Modelo carregado com sucesso!")

In [ ]:
# 1. Carregar Dataset
df_val = pd.read_csv("dataset_validacao.csv")
df_val['Text'] = df_val['Text'].astype(str)
df_val['label'] = df_val['Label'].map(label_mapping)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding='max_length', max_length=max_length)
        self.labels = labels.tolist()
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

val_loader = DataLoader(TextDataset(df_val['Text'], df_val['label'], tokenizer), batch_size=32)

In [ ]:
all_preds = []
all_labels = []

print("A processar validação...")
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        
        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# Cálculo de Métricas
acc = accuracy_score(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=unique_labels)

print(f"\n✅ ACCURACY GLOBAL: {acc:.4f}")
print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(report)

# Exportar resultados para CSV
df_val['predicted_label'] = all_preds
df_val['predicted_source'] = df_val['predicted_label'].map(reverse_mapping)
df_val.to_csv('resultados_validacao_deberta.csv', index=False)
print("Ficheiro 'resultados_validacao_deberta.csv' gerado.")

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score

print("A iniciar o teste com os dados da Submissão 1...")

In [ ]:
df_teste = pd.read_csv(os.path.join(DATA_DIR, "subm2.csv"), sep=';')
df_reais = pd.read_csv(os.path.join(DATA_DIR, "subm2_labels_revealed.csv"), sep=';')
df_teste = df_teste[df_teste['ID'].isin(df_reais['ID'])].copy()

In [ ]:
df_teste['Text'] = df_teste['Text'].astype(str)
print(f"Ficheiro cortado! Vamos processar apenas {len(df_teste)} textos.")

In [ ]:
class SubmissionDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding='max_length', max_length=max_length)
        
    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        
    def __len__(self):
        return len(self.encodings.input_ids)

test_loader = DataLoader(SubmissionDataset(df_teste['Text'], tokenizer), batch_size=32)

In [ ]:
# 3. Fazer as Previsões
model.eval()
test_preds = []

print("A pedir ao DeBERTa para adivinhar as fontes...")
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)

# Guardar as previsões do modelo
df_teste['predicted_label'] = test_preds
df_teste['predicted_source'] = df_teste['predicted_label'].map(reverse_mapping)

In [ ]:
# =====================================================================
# 4. COMPARAÇÃO COM OS RESULTADOS REAIS
# =====================================================================
print("\nA calcular a nota final...")

df_final = pd.merge(df_teste, df_reais, on='ID')

# A CORREÇÃO MÁGICA: .lower() para forçar tudo a letras minúsculas!
respostas_certas = [str(r).lower() for r in df_final['Label'].tolist()]
respostas_modelo = [str(r).lower() for r in df_final['predicted_source'].tolist()]

# 5. Calcular a nota
from sklearn.metrics import accuracy_score, classification_report

acc_teste = accuracy_score(respostas_certas, respostas_modelo)
report_teste = classification_report(respostas_certas, respostas_modelo)

print(f"\n🏆 ACCURACY DA SUBMISSÃO 1: {acc_teste:.4f}")
print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(report_teste)

In [ ]:
# Ver onde o modelo se espetou completamente!
df_final['Label_lower'] = df_final['Label'].str.lower()
df_final['Pred_lower'] = df_final['predicted_source'].str.lower()

# Filtrar apenas os erros
erros = df_final[df_final['Label_lower'] != df_final['Pred_lower']]

print(f"Total de erros encontrados: {len(erros)} em 100 textos.")

# Vamos olhar especificamente para os textos da Anthropic que ele falhou:
erros_anthropic = erros[erros['Label_lower'] == 'anthropic']

print("\n=== EXEMPLOS DE TEXTOS DA ANTHROPIC QUE ENGANARAM O MODELO ===")
for i, row in erros_anthropic.head(3).iterrows():
    print(f"\nVERDADE: {row['Label']} | O MODELO ACHOU QUE ERA: {row['predicted_source']}")
    print(f"TEXTO: {row['Text_x'][:300]}...") # Mostra os primeiros 300 caracteres